In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class PairsTradingBacktester:
    def __init__(self, symbol1, symbol2, initial_capital=100000, trade_amount=10000):
        """
        配對交易回測器
        
        Parameters:
        symbol1, symbol2: 兩隻股票代碼
        initial_capital: 初始資本
        trade_amount: 每次交易金額
        """
        self.symbol1 = symbol1
        self.symbol2 = symbol2
        self.initial_capital = initial_capital
        self.trade_amount = trade_amount
        self.data = None
        self.results = None
        
    def calculate_adj_close(self, data):
        """計算調整後收盤價"""
        try:
            # 如果已經有 Adj Close，直接使用
            if 'Adj Close' in data.columns:
                return data['Adj Close']
            
            # 如果沒有，則使用 Close, Open, High, Low 來計算
            close = data['Close']
            
            # 檢查是否有股息數據來進行調整
            if 'Dividends' in data.columns and data['Dividends'].sum() > 0:
                # 簡化的股息調整方法
                dividends = data['Dividends'].fillna(0)
                cumulative_dividends = dividends[::-1].cumsum()[::-1]  # 反向累積
                adj_close = close - cumulative_dividends
            else:
                # 如果沒有股息數據，使用收盤價作為調整後價格
                adj_close = close
                
            return adj_close
            
        except Exception as e:
            print(f"調整價格計算失敗，使用原始收盤價: {e}")
            return data['Close']
    
    def fetch_data(self, start_date='2022-01-01', end_date='2024-01-01'):
        """獲取股票數據"""
        print(f"正在獲取 {self.symbol1} 和 {self.symbol2} 的數據...")
        
        try:
            # 獲取兩隻股票的完整數據
            data1 = yf.download(self.symbol1, start=start_date, end=end_date)
            data2 = yf.download(self.symbol2, start=start_date, end=end_date)
            
            # 計算調整後收盤價
            adj_close1 = self.calculate_adj_close(data1)
            adj_close2 = self.calculate_adj_close(data2)
            
            # 合併數據
            self.data = pd.DataFrame({
                self.symbol1: adj_close1,
                self.symbol2: adj_close2
            }).dropna()
            
            print(f"成功獲取數據，共 {len(self.data)} 個交易日")
            print(f"使用調整後收盤價進行分析")
            return True
            
        except Exception as e:
            print(f"數據獲取失敗: {e}")
            return False
    
    def calculate_spread(self, lookback_period=30):
        """計算價差和Z-score"""
        if self.data is None:
            print("請先獲取數據")
            return
        
        # 計算對數價格
        log_price1 = np.log(self.data[self.symbol1])
        log_price2 = np.log(self.data[self.symbol2])
        
        # 計算滾動相關係數來確定對沖比率
        correlation = log_price1.rolling(window=lookback_period).corr(log_price2)
        
        # 簡化的對沖比率計算（使用價格比率）
        hedge_ratio = (self.data[self.symbol1] / self.data[self.symbol2]).rolling(window=lookback_period).mean()
        
        # 計算價差
        spread = log_price1 - hedge_ratio * log_price2
        
        # 計算Z-score
        spread_mean = spread.rolling(window=lookback_period).mean()
        spread_std = spread.rolling(window=lookback_period).std()
        z_score = (spread - spread_mean) / spread_std
        
        # 保存結果
        self.data['hedge_ratio'] = hedge_ratio
        self.data['spread'] = spread
        self.data['z_score'] = z_score
        self.data['correlation'] = correlation
        
        print("價差和Z-score計算完成")
    
    def generate_signals(self, entry_threshold=2.0, exit_threshold=0.5):
        """生成交易信號"""
        if 'z_score' not in self.data.columns:
            print("請先計算價差")
            return
        
        signals = pd.DataFrame(index=self.data.index)
        signals['signal'] = 0
        
        # 當Z-score > entry_threshold時，做空價差（買入股票2，賣出股票1）
        # 當Z-score < -entry_threshold時，做多價差（買入股票1，賣出股票2）
        # 當Z-score接近0時，平倉
        
        position = 0
        for i in range(1, len(self.data)):
            z_current = self.data['z_score'].iloc[i]
            
            if position == 0:  # 沒有持倉
                if z_current > entry_threshold:
                    signals['signal'].iloc[i] = -1  # 做空價差
                    position = -1
                elif z_current < -entry_threshold:
                    signals['signal'].iloc[i] = 1   # 做多價差
                    position = 1
            
            elif position == 1:  # 持有多頭價差
                if z_current > -exit_threshold:
                    signals['signal'].iloc[i] = 0   # 平倉
                    position = 0
            
            elif position == -1:  # 持有空頭價差
                if z_current < exit_threshold:
                    signals['signal'].iloc[i] = 0   # 平倉
                    position = 0
        
        self.data['signal'] = signals['signal']
        print(f"交易信號生成完成，共產生 {len(signals[signals['signal'] != 0])} 個信號")
    
    def backtest(self):
        """執行回測"""
        if 'signal' not in self.data.columns:
            print("請先生成交易信號")
            return
        
        # 初始化結果DataFrame
        results = pd.DataFrame(index=self.data.index)
        results['price1'] = self.data[self.symbol1]
        results['price2'] = self.data[self.symbol2]
        results['signal'] = self.data['signal']
        results['position'] = 0
        results['cash'] = self.initial_capital
        results['holdings1'] = 0  # 股票1持股數量
        results['holdings2'] = 0  # 股票2持股數量
        results['portfolio_value'] = self.initial_capital
        
        # 執行交易
        for i in range(1, len(results)):
            # 複製前一天的狀態
            results['position'].iloc[i] = results['position'].iloc[i-1]
            results['cash'].iloc[i] = results['cash'].iloc[i-1]
            results['holdings1'].iloc[i] = results['holdings1'].iloc[i-1]
            results['holdings2'].iloc[i] = results['holdings2'].iloc[i-1]
            
            signal = results['signal'].iloc[i]
            price1 = results['price1'].iloc[i]
            price2 = results['price2'].iloc[i]
            
            if signal != 0 and results['position'].iloc[i-1] == 0:
                # 開倉
                hedge_ratio = self.data['hedge_ratio'].iloc[i]
                
                if signal == 1:  # 做多價差：買入股票1，賣出股票2
                    shares1 = self.trade_amount / price1
                    shares2 = shares1 * hedge_ratio
                    
                    cost = shares1 * price1 - shares2 * price2
                    
                    if results['cash'].iloc[i] >= cost:
                        results['holdings1'].iloc[i] = shares1
                        results['holdings2'].iloc[i] = -shares2
                        results['cash'].iloc[i] -= cost
                        results['position'].iloc[i] = 1
                
                elif signal == -1:  # 做空價差：賣出股票1，買入股票2
                    shares1 = self.trade_amount / price1
                    shares2 = shares1 * hedge_ratio
                    
                    cost = -shares1 * price1 + shares2 * price2
                    
                    if results['cash'].iloc[i] >= cost:
                        results['holdings1'].iloc[i] = -shares1
                        results['holdings2'].iloc[i] = shares2
                        results['cash'].iloc[i] -= cost
                        results['position'].iloc[i] = -1
            
            elif signal == 0 and results['position'].iloc[i-1] != 0:
                # 平倉
                proceeds1 = results['holdings1'].iloc[i] * price1
                proceeds2 = results['holdings2'].iloc[i] * price2
                
                results['cash'].iloc[i] += (proceeds1 + proceeds2)
                results['holdings1'].iloc[i] = 0
                results['holdings2'].iloc[i] = 0
                results['position'].iloc[i] = 0
            
            # 計算投資組合總價值
            portfolio_value = (results['cash'].iloc[i] + 
                             results['holdings1'].iloc[i] * price1 + 
                             results['holdings2'].iloc[i] * price2)
            results['portfolio_value'].iloc[i] = portfolio_value
        
        # 計算收益率
        results['returns'] = results['portfolio_value'].pct_change()
        results['cumulative_returns'] = (1 + results['returns']).cumprod() - 1
        
        self.results = results
        print("回測完成")
    
    def calculate_performance_metrics(self):
        """計算績效指標"""
        if self.results is None:
            print("請先執行回測")
            return
        
        returns = self.results['returns'].dropna()
        
        # 基本指標
        total_return = (self.results['portfolio_value'].iloc[-1] / self.initial_capital - 1) * 100
        annual_return = ((self.results['portfolio_value'].iloc[-1] / self.initial_capital) ** 
                        (252 / len(returns)) - 1) * 100
        annual_volatility = returns.std() * np.sqrt(252) * 100
        sharpe_ratio = annual_return / annual_volatility if annual_volatility != 0 else 0
        
        # 最大回撤
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min() * 100
        
        # 交易統計
        trades = len(self.results[self.results['signal'] != 0])
        
        metrics = {
            '總收益率 (%)': round(total_return, 2),
            '年化收益率 (%)': round(annual_return, 2),
            '年化波動率 (%)': round(annual_volatility, 2),
            '夏普比率': round(sharpe_ratio, 2),
            '最大回撤 (%)': round(max_drawdown, 2),
            '交易次數': trades,
            '最終資產': round(self.results['portfolio_value'].iloc[-1], 2)
        }
        
        return metrics
    
    def plot_results(self):
        """繪製結果圖表"""
        if self.results is None:
            print("請先執行回測")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. 股價走勢
        axes[0, 0].plot(self.data.index, self.data[self.symbol1], label=self.symbol1, alpha=0.7)
        axes[0, 0].plot(self.data.index, self.data[self.symbol2], label=self.symbol2, alpha=0.7)
        axes[0, 0].set_title('股價走勢')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Z-score和交易信號
        axes[0, 1].plot(self.data.index, self.data['z_score'], label='Z-score', color='blue')
        axes[0, 1].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='進場門檻')
        axes[0, 1].axhline(y=-2, color='red', linestyle='--', alpha=0.7)
        axes[0, 1].axhline(y=0.5, color='green', linestyle='--', alpha=0.7, label='出場門檻')
        axes[0, 1].axhline(y=-0.5, color='green', linestyle='--', alpha=0.7)
        axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
        
        # 標記交易點
        buy_signals = self.results[self.results['signal'] == 1].index
        sell_signals = self.results[self.results['signal'] == -1].index
        
        if len(buy_signals) > 0:
            axes[0, 1].scatter(buy_signals, self.data.loc[buy_signals, 'z_score'], 
                             color='green', marker='^', s=50, label='做多信號')
        if len(sell_signals) > 0:
            axes[0, 1].scatter(sell_signals, self.data.loc[sell_signals, 'z_score'], 
                             color='red', marker='v', s=50, label='做空信號')
        
        axes[0, 1].set_title('Z-score和交易信號')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. 權益曲線
        axes[1, 0].plot(self.results.index, self.results['portfolio_value'], 
                       color='purple', linewidth=2, label='投資組合價值')
        axes[1, 0].axhline(y=self.initial_capital, color='gray', linestyle='--', alpha=0.7)
        axes[1, 0].set_title('權益曲線 (Equity Curve)')
        axes[1, 0].set_ylabel('投資組合價值')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. 累積收益率
        axes[1, 1].plot(self.results.index, self.results['cumulative_returns'] * 100, 
                       color='orange', linewidth=2)
        axes[1, 1].set_title('累積收益率 (%)')
        axes[1, 1].set_ylabel('收益率 (%)')
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    def run_backtest(self, start_date='2022-01-01', end_date='2024-01-01', 
                     entry_threshold=2.0, exit_threshold=0.5, lookback_period=30):
        """執行完整的回測流程"""
        print(f"開始配對交易回測：{self.symbol1} vs {self.symbol2}")
        print(f"初始資本：${self.initial_capital:,}")
        print(f"每次交易金額：${self.trade_amount:,}")
        print("-" * 50)
        
        # 1. 獲取數據
        if not self.fetch_data(start_date, end_date):
            return
        
        # 2. 計算價差
        self.calculate_spread(lookback_period)
        
        # 3. 生成信號
        self.generate_signals(entry_threshold, exit_threshold)
        
        # 4. 執行回測
        self.backtest()
        
        # 5. 計算績效
        metrics = self.calculate_performance_metrics()
        
        # 6. 顯示結果
        print("\n績效指標：")
        print("-" * 30)
        for key, value in metrics.items():
            print(f"{key}: {value}")
        
        # 7. 繪製圖表
        self.plot_results()
        
        return metrics

In [17]:
# 使用範例
if __name__ == "__main__":
    # 創建回測器實例
    # 例如：測試可口可樂(KO)和百事可樂(PEP)的配對交易
    backtester = PairsTradingBacktester(
        symbol1='KO',      # 可口可樂
        symbol2='PEP',     # 百事可樂
        initial_capital=100000,  # 初始資本10萬美元
        trade_amount=10000       # 每次交易1萬美元
    )
    
    # 執行回測
    results = backtester.run_backtest(
        start_date='2022-01-01',
        end_date='2024-01-01',
        entry_threshold=2.0,    # Z-score進場門檻
        exit_threshold=0.5,     # Z-score出場門檻
        lookback_period=30      # 計算統計量的回望期
    )

[*********************100%***********************]  1 of 1 completed

開始配對交易回測：KO vs PEP
初始資本：$100,000
每次交易金額：$10,000
--------------------------------------------------
正在獲取 KO 和 PEP 的數據...



[*********************100%***********************]  1 of 1 completed

數據獲取失敗: If using all scalar values, you must pass an index
